# PSet #3 · Notebook 06 — Verificación independiente del Champion

**Ejecutar con un kernel nuevo** usando `Restart Kernel and Run All Cells`. Este notebook no entrena ni modifica el modelo: lo recupera del Model Registry mediante `models:/...@champion` y reproduce el AP de test que el notebook 05 guardó y registró.

In [1]:
from pathlib import Path
import json, os
import pandas as pd
import numpy as np
import mlflow, mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from src.datos import leer_obt, variables_modelo, hash_snapshot, ORIGEN

OUT=Path('/workspace/docs')
mlflow.set_tracking_uri(os.environ.get('MLFLOW_TRACKING_URI','http://mlflow:5000'))
mlflow.set_registry_uri(os.environ.get('MLFLOW_TRACKING_URI','http://mlflow:5000'))
sel = json.loads((OUT/'seleccion_final.json').read_text(encoding='utf-8'))
info = json.loads((OUT/'split_info.json').read_text(encoding='utf-8'))
df = leer_obt().sort_values('CUSTOMER_ID',kind='stable').reset_index(drop=True)
assert hash_snapshot(df) == info['hash_dataset'] == sel['dataset_hash']
X,y=variables_modelo(df)
idx_train,idx_test=train_test_split(np.arange(len(df)),test_size=.20,
                                   random_state=int(info['seed']),stratify=y)
idx_test=np.sort(idx_test)
manifest=pd.read_csv(OUT/'split_train_test.csv',dtype={'CUSTOMER_ID':str})
assert manifest['CUSTOMER_ID'].tolist() == df['CUSTOMER_ID'].tolist()
assert (manifest.iloc[idx_test]['split']=='test').all()
uri=f"models:/{sel['registry_name']}@champion"
champion=mlflow.sklearn.load_model(uri)
scores=champion.predict_proba(X.iloc[idx_test])[:,1]
medido=float(average_precision_score(y.iloc[idx_test],scores))
registrado=float(sel['modelos']['champion']['test_ap'])
assert np.isclose(medido,registrado,atol=1e-12,rtol=0),(
    f'AP cambiado: {medido} vs {registrado}')
print('URI:',uri)
print('Snapshot:',info['hash_dataset'])
print('Holdout:',len(idx_test),'clientes')
print('AP calculado:',repr(medido))
print('AP guardado:',repr(registrado))
print('RESULTADO: MODELO CHAMPION REPRODUCIBLE')

URI: models:/PSet3_Churn_Telco-modelo@champion
Snapshot: d394e39192c935f813db937647ea608341131d5b3021696fbbb186daff7043d3
Holdout: 1409 clientes
AP calculado: 0.6614154005251454
AP guardado: 0.6614154005251454
RESULTADO: MODELO CHAMPION REPRODUCIBLE
